# 08 · Selection Bias nach Zheng et al. — Originalprotokoll für offene Modelle

**Ziel:** Die im veröffentlichten Open-Model-Code enthaltenen Bedingungen der
Selection-Bias-Analyse nachvollziehbar ausführen und auswerten. Die Prompt- und
Scoring-Funktionen werden **unverändert aus dem fixierten Originalcode geladen**,
nicht durch unsere bisherige Vier-Optionen-Pipeline ersetzt.

Das Notebook führt nur CPU-Beispiele, Protokollprüfungen und Auswertungen aus.
Echte Modellläufe werden mit dem dazugehörigen Slurm-Auftrag gestartet.

**Grenze von „1:1“:** Versuchsbedingungen und Funktionen entsprechen dem
veröffentlichten Open-Model-Protokoll; neuere Modelle, Tokenizer, Pakete und GPU-
Numerik sind kein bitgenauer Nachbau der historischen Experimente. GPT-3.5-API-
Experimente (alternative Labels, CoT und zusätzliche Debiasing-Instruktion),
die 2-/3-Optionen-Zusatzanalysen und die vollständige Wiederholung aller
20 damaligen Modelle sind hier nicht als implementiert ausgewiesen.

Quellen:
- [Paper, Abschnitt 2 und Anhang E](https://proceedings.iclr.cc/paper_files/paper/2024/file/54dd9e0cff6d9214e20d97eb2a3bae49-Paper-Conference.pdf)
- [Originaler Versuchsplan](https://github.com/chujiezheng/LLM-MCQ-Bias/blob/2ae2f40c77006f7a00a4675bdfb30151c2406691/code/scripts/run_llama-7b.sh)
- [Originale Prompts und Scorer](https://github.com/chujiezheng/LLM-MCQ-Bias/blob/2ae2f40c77006f7a00a4675bdfb30151c2406691/code/eval_clm_utils.py)

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
                     for p in (base, base / "analysis")
                     if (p / "src").is_dir() and (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Notebook im MCQ-Projekt öffnen.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.experimental.zheng_selection_bias import (
    COMMIT, HASHES, REFERENCE, load_original, prepared, original_settings, summarize,
)
original = load_original()  # Prüft SHA-256 der Originalfunktionen; lädt kein Modell.
print("Original-Commit:", COMMIT)
print("Originaldateien geprüft:", ", ".join(HASHES))

Original-Commit: 2ae2f40c77006f7a00a4675bdfb30151c2406691
Originaldateien geprüft: mmlu_categories, utils, eval_clm_utils


## 1 · Daten und exakte Evaluationspopulation

Die unveränderten CSVs stammen aus dem Originalrepository. Der Originalcode
filtert referenzierende Antwortoptionen erst bei der Auswertung; hier wird
**dieselbe Regel** verwendet. Die medizinische Preprocessing-Policy wird nicht
zusätzlich angewendet. MMLU wird zusätzlich in die vier Originaldomänen zerlegt.

CSQA hat **fünf** Optionen. Der Repository-Testsplit besteht aus 1.216 gelabelten
Validation-Fragen; fünf weitere wurden mit Seed 23 für Few-Shot reserviert.
Dies ist nicht der versteckte offizielle CSQA-Testsplit.

In [2]:
manifest = json.loads((REFERENCE / "manifest.json").read_text())
inventory = pd.DataFrame([
    {"Datensatz": d["dataset"], "Split im Repository": d["repository_split"],
     "Originalzeilen": d["original_rows"], "Ausgeschlossen": d["upstream_filter_excluded"],
     "Auswertbar": d["retained_rows"], "Optionen": d["options"]}
    for d in manifest["datasets"]
])
display(inventory)
assert inventory.query("`Split im Repository` == 'test'")["Auswertbar"].tolist() == [13592, 1165, 1216]

,Datensatz,Split im Repository,Originalzeilen,Ausgeschlossen,Auswertbar,Optionen
0,mmlu,dev,285,5,280,4
1,mmlu,test,14042,450,13592,4
2,arc,dev,5,0,5,4
3,arc,test,1165,0,1165,4
4,csqa,dev,5,0,5,5
5,csqa,test,1216,0,1216,5


## 2 · Der veröffentlichte Versuchsplan

Für jeden Datensatz und jedes Modell werden **0-shot und 5-shot** getrennt
untersucht. Bei 5-shot kommen die fünf Beispiele aus der jeweiligen Dev-Datei
(bei MMLU fachbezogen). Der Originalrunner verwendet rohe Textprompts, keine
modernen Chat-Templates. Der Input wird auf die letzten **1.536 Tokens** begrenzt.

| Bedingung | Was unverändert aus dem Original übernommen wird |
|---|---|
| `perm` / `cyclic` | 24 Permutationen bei MMLU/ARC; fünf zyklische bei CSQA |
| Baseline | Identitätsreihenfolge, also Index 0 der gespeicherten Permutationen |
| `move_a` … `move_d` | MMLU: richtige Antwort durch **Tausch** an die Zielposition bringen |
| `shuffle_both` | Label-Inhalt-Paare gemeinsam mischen; Labels bleiben an ihrem Inhalt |
| `noid` | Labels entfernen und vollständige Antworttexte nach längennormierter Likelihood bewerten |

„Answer Moving“ ist kein zyklisches Verschieben. „Shuffling IDs“ erhält die
Zuordnung von Buchstabe zu Inhalt, entkoppelt sie aber von der dargestellten Position.
Die 24 Permutationen sind teurer als unsere bisherige Pipeline mit nur vier
zyklischen Permutationen; der Referenzauftrag bildet den veröffentlichten Plan ab.

In [3]:
plan = pd.DataFrame([
    {"Datensatz": dataset, "Shots": shots, "Bedingung": setting}
    for dataset in ["mmlu", "arc", "csqa"] for shots in [0, 5]
    for setting in original_settings(dataset)
])
display(plan)
print("Bedingungen je Modell:", len(plan))

,Datensatz,Shots,Bedingung
0,mmlu,0,perm
1,mmlu,0,noid
2,mmlu,0,shuffle_both
3,mmlu,0,move_a
4,mmlu,0,move_b
5,mmlu,0,move_c
6,mmlu,0,move_d
7,mmlu,5,perm
8,mmlu,5,noid
9,mmlu,5,shuffle_both


Bedingungen je Modell: 26


## 3 · Originalprompts und Antwortzuordnung ansehen

Die nächste Zelle ruft die Originalfunktionen für eine Beispiel-Frage auf.
Sie lädt keine Modellgewichte. Eine temporäre Arbeitsumgebung verhindert,
dass der Originalcode Ergebnisordner in den Referenzdaten anlegt.

In [4]:
examples = []
for setting in ["default", "move_a", "shuffle_both", "noid"]:
    with prepared(original, "mmlu", 0, setting) as (subjects, _, samples_fn, _):
        subject = subjects[0]
        prompt_parts, options, gold = samples_fn(subject)[0]
        examples.append({"Bedingung": setting, "Goldlabel": gold,
                         "Systemtext": prompt_parts[0], "Fragetext": prompt_parts[1]})
for example in examples:
    print("\n---", example["Bedingung"], "| richtig:", example["Goldlabel"], "---")
    print(example["Systemtext"] + "\n\n" + example["Fragetext"])


--- default | richtig: B ---
The following are multiple choice questions about abstract algebra. You should directly answer the question by choosing the correct option.

Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Options:
A. 0
B. 4
C. 2
D. 6
Answer:

--- move_a | richtig: A ---
The following are multiple choice questions about abstract algebra. You should directly answer the question by choosing the correct option.

Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Options:
A. 4
B. 0
C. 2
D. 6
Answer:

--- shuffle_both | richtig: B ---
The following are multiple choice questions about abstract algebra. You should directly answer the question by choosing the correct option.

Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Options:
A. 0
C. 2
D. 6
B. 4
Answer:

--- noid | richtig: B ---
The following are multiple choice questions about abstract alge

## 4 · First Token: die Originalbehandlung der Leerzeichen

Hier besteht ein wichtiger Unterschied zu unserem bisherigen Scorer:
Zheng bestimmt je Label zwei Kandidaten über das letzte Token von `': A'` und
`':A'` (entsprechend B–E). Er berechnet Softmax über die **2 × Anzahl Optionen**
Einträge und addiert anschließend beide Varianten pro Label.
Doppelte Token-IDs bleiben wie im Original erhalten. Ob der Prompt ein zusätzliches
Leerzeichen erhält, entscheidet der originale Tokenizer-Test.

Das folgende Beispiel zeigt nur die Mathematik mit **synthetischen Logits**.
Die echten Jobausgaben speichern die Token-IDs, die einzelnen Logits sowie die
originalen resultierenden Wahrscheinlichkeiten. Unser bisheriges Ein-Token-pro-
Label-Verfahren wird für diesen Referenzauftrag nicht eingesetzt.

In [5]:
labels = list("ABCD")
# Erste Zeile: mit Leerzeichen; zweite: ohne. Reines Rechenbeispiel!
slot_logits = np.array([[2.0, 1.0, 0.5, -0.2], [1.5, 1.2, 0.1, -0.1]])
slot_probs = np.exp(slot_logits - slot_logits.max())
slot_probs /= slot_probs.sum()
option_probs = slot_probs.sum(axis=0)
display(pd.DataFrame({"Label": labels, "Logit mit Leerzeichen": slot_logits[0],
                      "Logit ohne Leerzeichen": slot_logits[1],
                      "Summierte Wahrscheinlichkeit": option_probs}))
assert np.isclose(option_probs.sum(), 1)
print("Synthetische Vorhersage:", labels[option_probs.argmax()])

,Label,Logit mit Leerzeichen,Logit ohne Leerzeichen,Summierte Wahrscheinlichkeit
0,A,2.0,1.5,0.530261
1,B,1.0,1.2,0.269733
2,C,0.5,0.1,0.123015
3,D,-0.2,-0.1,0.076991


Synthetische Vorhersage: A


## 5 · Tokenbias und Positionsbias: zwei Ablationen

**Shuffling IDs:** Der Originalcode mischt Label-Inhalt-Paare mit einem aus den
Antworttexten abgeleiteten Seed. Die Optionen stehen danach z. B. in der Reihenfolge
B/D/A/C. Das richtige Label ändert sich nicht. Der Bericht zählt Labels und
angezeigte Positionen getrennt; diese beiden Verteilungen dürfen nicht verwechselt
werden. Die zusätzliche Positionszählung ist unsere diagnostische Ergänzung.

**Removing IDs:** Es stehen nur die Antworttexte im Prompt. Für jede Option wird
separat die mittlere negative Log-Likelihood ihrer Fortsetzung berechnet. Die
Option mit der kleinsten mittleren Loss wird gewählt. Dies ist **kein** First-
Token-A/B/C/D-Scoring. Der Referenzrunner nutzt dafür unverändert
`prepare_eval_fn_noid`, einschließlich Maskierung und Trunkierung.

Die Ablationen prüfen die Hypothesen aus Abschnitt 2.4. Da sie auch Prompt und
Antwortformat ändern, ist ihre Differenz keine exakte additive Zerlegung von
„Gesamtbias = Tokenbias + Positionsbias“.

In [6]:
base_options = ["Inhalt eins", "Inhalt zwei", "Inhalt drei", "Inhalt vier"]
shuffled_labels, shuffled_options = original.utils.shuffle_options_with_ids(labels, base_options)
display(pd.DataFrame({"Angezeigte Position": range(1, 5),
                      "Label": shuffled_labels, "Inhalt": shuffled_options}))
assert dict(zip(shuffled_labels, shuffled_options)) == dict(zip(labels, base_options))
# Synthetische mittlere Token-Loss, nur zur Erklärung von noid.
mean_losses = np.array([1.7, 0.8, 1.1, 1.9])
likelihood_weights = np.exp(-mean_losses - (-mean_losses).max())
likelihood_weights /= likelihood_weights.sum() + 1e-10
print("noid-Demo: gewählt wird", base_options[mean_losses.argmin()])
display(pd.DataFrame({"Optionstext": base_options, "Mittlere Loss (Demo)": mean_losses,
                      "Normalisierte Gewichte": likelihood_weights}))

,Angezeigte Position,Label,Inhalt
0,1,A,Inhalt eins
1,2,B,Inhalt zwei
2,3,D,Inhalt vier
3,4,C,Inhalt drei


noid-Demo: gewählt wird Inhalt zwei


,Optionstext,Mittlere Loss (Demo),Normalisierte Gewichte
0,Inhalt eins,1.7,0.163922
1,Inhalt zwei,0.8,0.403184
2,Inhalt drei,1.1,0.298686
3,Inhalt vier,1.9,0.134208


## 6 · Metriken wie im Paper

Ausgewertet werden Accuracy und Recall je Label/Originalposition sowie RStd,
die Populationsstandardabweichung der Recalls (`ddof=0`). Hier stehen wie im
Originalbericht alle Werte auf der **Prozentskala**. RStd=5 bedeutet daher
5 Prozentpunkte; unsere bisherigen Pipeline-CSVs verwenden dagegen 0–1.

Für die Darstellung analog zu Figure 3 werden die Recalls um die Gesamtaccuracy
zentriert (`Recall − Accuracy`). Für „Answer Moving“ ist die Accuracy pro
Zielposition relevant: Da dann nur eine Goldklasse vorkommt, ist RStd über alle
Klassen nicht bestimmbar. Es wird dafür kein künstlicher Nullwert eingesetzt.

Bei `noid` bezeichnet das intern gespeicherte A/B/C/D/E die ursprüngliche Option;
das Modell hat diese Labels nicht gesehen. Die Originalfilter werden erst auf
die gespeicherten Vorhersagen angewendet. Bei `perm`/`cyclic` liefert Index 0 die
Default-Baseline, nicht der Mittelwert aller Permutationen.

In [7]:
# Ein überprüfbares synthetisches Beispiel; keine echten Modellresultate.
demo_records = [
    {"shots": 0, "setting": "default", "subject": "arc", "data": {
        "idx": i, "options": base_options, "ideal": label, "sampled": "A"}}
    for i, label in enumerate(labels)
]
demo_summary = summarize(demo_records, "arc", original)
display(demo_summary[["n", "accuracy", "recall_A", "recall_B", "recall_C", "recall_D", "rstd"]])
assert demo_summary.iloc[0].accuracy == 25
assert np.isclose(demo_summary.iloc[0].rstd, np.std([100, 0, 0, 0]))

,n,accuracy,recall_A,recall_B,recall_C,recall_D,rstd
0,4,25.0,100.0,0.0,0.0,0.0,43.30127


## 7 · Modellläufe auf LRZ einreichen

Der Aufruf aus dem folgenden Text wird **nicht im Notebook ausgeführt**.
Zuerst die aktualisierten `scripts/`, `src/`, `configs/` und die Referenzdaten
nach LRZ übertragen. Dann dort die vorhandene Umgebung aktivieren. Ein erster
Test verwendet insgesamt zwei ausgewählte Fragen, dieselben für jede Bedingung:

```bash
cd /dss/dsshome1/0F/ra39dik2/analysis
export MCQ_CONDA_ENV=mcq-analysis
export PARTITION=lrz-dgx-a100-80x8
export MODEL=qwen2.5-7b-instruct
export DATASET=arc
export LIMIT=2
unset RESUME
bash scripts/slurm/submit.sh zheng
```

Partition zuvor mit `sinfo` prüfen. Der Job durchläuft 0-/5-shot und die
Originalbedingungen. Er trainiert kein Modell. Nach geprüftem Erfolg wird ein
vollständiger Lauf mit `unset LIMIT` eingereicht. Je Modell und Datensatz ein
separater Auftrag. Für Voll-Läufe sind fixierte Modell- und Tokenizerrevisionen
verpflichtend; Qwen3s bisheriges `main` reicht dafür nicht.

`run_zheng_selection_bias.py --plan-only --dataset arc` zeigt den Plan ohne
Modellstart. `--resume outputs/zheng_selection_bias/<run>` setzt einen abgebrochenen
Lauf mit identischen Argumenten fort. Die Resume-Prüfung berücksichtigt auch die
Paketstände und Modellrevisionen. Slurm-Logs: `mcq-zheng-<job-id>.out/.err`.

Dateien je Lauf: `manifest.json` (Protokoll/Revisionen/Token-IDs), `records.jsonl`
(Prompts, Wahrscheinlichkeiten, Logits bzw. Loss), `summary.csv` (Metriken).
Dieser Referenzauftrag ist von der bisherigen medizinischen PriDe-Pipeline getrennt.

## 8 · Gespeicherte Ergebnisse auswählen und prüfen

Trage explizit einen Run-Ordner pro Modell und Datensatz ein. Standardmäßig sind
nur vollständige, unlimitierte Läufe erlaubt. Ohne Ergebnisse bleibt der Bericht
leer. Die vorigen Beispiele werden nie als echte Resultate übernommen.

In [8]:
RUN_DIRS = []  # Beispiel: PROJECT_ROOT / "outputs/zheng_selection_bias/<run-id>"
ALLOW_LIMITED = False
result_frames = []
seen = set()
for folder in RUN_DIRS:
    folder = Path(folder)
    state = json.loads((folder / "manifest.json").read_text())
    config = state["config"]
    if state["status"] != "completed":
        raise ValueError(f"Lauf nicht abgeschlossen: {folder}")
    if config["upstream_commit"] != COMMIT or config["source_hashes"] != HASHES:
        raise ValueError("Abweichende Originalversion")
    if config.get("scorer_backend") != "real_transformers":
        raise ValueError("Keine echten Modellresultate")
    if config["limit"] is not None and not ALLOW_LIMITED:
        raise ValueError("Kleiner Testlauf: für technische Ansicht ALLOW_LIMITED=True setzen")
    key = (config["model"], config["dataset"])
    if key in seen:
        raise ValueError("Mehrere Läufe derselben Modell-Datensatz-Kombination ausgewählt")
    seen.add(key)
    frame = pd.read_csv(folder / "summary.csv")
    frame["model"] = config["model"]
    frame["run_dir"] = str(folder)
    result_frames.append(frame)
results = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()
if results.empty:
    print("Noch keine echten Läufe ausgewählt. Keine Ergebniswerte oder Plots erzeugt.")
else:
    display(results)

Noch keine echten Läufe ausgewählt. Keine Ergebniswerte oder Plots erzeugt.


## 9 · Tabellen und Figuren der Selection-Bias-Analyse

Die erste Tabelle vergleicht Default, Shuffling IDs und Removing IDs. Die zweite
zeigt Original vs. Move-to-A/B/C/D für MMLU (Aufbau wie Tabelle 1 des Papers).
Die Grafik stellt zentrierte Recalls nach dem Prinzip von Figure 3 dar, einschließlich
der MMLU-Domänen. Es werden keine historischen Paperzahlen als eigene Messungen
übernommen. Für neuere Modelle bleibt „besser“ eine zu prüfende Hypothese.

In [9]:
import matplotlib.pyplot as plt

if not results.empty:
    report = results.copy()
    report["method"] = report.setting.replace({"perm": "Default", "cyclic": "Default",
        "default": "Default", "shuffle_both": "Shuffling IDs", "noid": "Removing IDs"})
    subset = report[report.method.isin(["Default", "Shuffling IDs", "Removing IDs"])]
    if subset.duplicated(["model", "dataset", "shots", "domain", "method"]).any():
        raise ValueError("Mehrere Default-Auswertungen; nur ein Protokoll pro Vergleich wählen")
    display(subset[["model", "dataset", "shots", "domain", "method", "n", "accuracy", "rstd"]])
    moving = report[(report.dataset == "mmlu") & (report.domain == "Overall") &
                    report.method.isin(["Default", "move_a", "move_b", "move_c", "move_d"])]
    if not moving.empty:
        display(moving.pivot(index=["model", "shots"], columns="method", values="accuracy"))
    for (model, dataset, shots), frame in subset.groupby(["model", "dataset", "shots"]):
        domains = list(frame.domain.unique())
        fig, axes = plt.subplots(1, len(domains), figsize=(4 * len(domains), 4),
                                 squeeze=False, layout="constrained")
        labels_for_data = list("ABCDE" if dataset == "csqa" else "ABCD")
        for ax, domain in zip(axes[0], domains):
            for _, row in frame[frame.domain == domain].iterrows():
                ax.plot(labels_for_data,
                        [row[f"recall_{label}"] - row.accuracy for label in labels_for_data],
                        marker="o", label=row.method)
            ax.axhline(0, color="gray", linewidth=0.8)
            ax.set(title=domain, xlabel="Option-ID / ursprüngliche Option bei noid",
                   ylabel="Recall − Accuracy (Prozentpunkte)")
            ax.legend(fontsize=8)
        fig.suptitle(f"{'TECHNISCHER TEST · ' if ALLOW_LIMITED else ''}{model} · {dataset} · {shots}-shot")
        plt.show()

## 10 · Einordnung und nächster Schritt

Diese Auswertung beantwortet zunächst, wie stark Selection Bias ist und wie er
sich bei ID-Shuffling und ID-Entfernung verändert. Die Permutationswahrscheinlichkeiten
können anschließend für PriDe genutzt werden. **Dieser Auftrag führt PriDe selbst
noch nicht aus**; Notebook 06 beschreibt die bestehende Vier-Optionen-PriDe-Pipeline.
Eine durchgehende Fünf-Optionen-PriDe-Pipeline ist davon getrennt zu ergänzen.

Für die Thesis: RQ1 kann das Originalprotokoll mit neueren Modellen untersuchen;
RQ2 erweitert auf medizinische Daten. Neue Chat-Templates oder andere Prompts
gehören in eine ausdrücklich benannte weitere Bedingung, nicht unbemerkt in den
Referenzlauf. Paper-übergreifende Unterschiede können außer Modellgeneration
auch Modellgröße, Training und Software betreffen.

Technische Referenz: `src/experimental/zheng_selection_bias.py`,
`scripts/run_zheng_selection_bias.py`, `scripts/slurm/zheng_selection_bias.sbatch`.
Die Tests prüfen Originalscoring mit beiden Tokenvarianten, 4-/5-Optionen,
0-/5-shot, No-ID-Likelihood, Answer Moving, Originalfilter und Recall/RStd.